In [15]:
import S1GRD_ImageUtils as iu 
import ee
import geemap
from typing import Literal
import math

In [31]:

from numpy.char import upper


def _get_srtm() -> ee.Image:
    """
    Returns a one band image containing values from the NASA SRTM Digital Elevation 30m
    """
    dataset = ee.Image('USGS/SRTMGL1_003')
    return dataset.select('elevation')

def _get_azimuth_relative_to_image_grid(angle: ee.Image) -> ee.Number:
      """
      Takes an ee.Image with one band containing the indidence angle values
      (relative to earth elipse).
      Returns the direection relative to the top of the image that the image
      was taken from.
      """
      return ee.Number(
                ee.Terrain.aspect(angle) \
                .reduceRegion(ee.Reducer.mean(), angle.geometry(), 100) \
                .get('aspect')
                )

def _get_azimuth_image_relative_to_image_grid(angle: ee.Image) -> ee.Image:
      """
      Takes an ee.Image with one band containing the indidence angle values
      (relative to earth elipse).
      Returns the direction relative to the top of the image that the image
      was taken from. For each pixel seperately.
      """
      return ee.Image(ee.Terrain.aspect(angle).select('aspect'))

      
def _get_image_corners(image) -> ee.Dictionary:
    """
    Returns the four extreme corner points of a given image's geometry
    as (lon, lat) pairs: the most northern, eastern, southern, and
    western points.
    """
    # Ring repeats its first point at the end to close the polygon - drop it
    # so it can't be mistaken for a distinct corner when sorting.
    ring = ee.List(image.geometry().coordinates().get(0))
    coords = ring.slice(0, ring.length().subtract(1))
    coords = ee.Array(coords).transpose()

    all_longitudes = ee.List(coords.toList().get(0))
    all_latitudes = ee.List(coords.toList().get(1))
    paired = all_longitudes.zip(all_latitudes)  # [[lon, lat], ...]

    # Sort once by each axis; the extremes sit at the ends of each sort.
    by_longitude = paired.sort(all_longitudes)
    by_latitude = paired.sort(all_latitudes)

    west = by_longitude.get(0)
    east = by_longitude.get(-1)
    south = by_latitude.get(0)
    north = by_latitude.get(-1)

    return ee.Dictionary({
        'north': north,
        'east': east,
        'south': south,
        'west': west,
    })

def _get_image_tilt(image: ee.Image, pass_type: Literal["ASCENDING", "DESCENDING"]) -> ee.Number:
        """
        Computes the tilt of the image relative to North, using the two most
        eastern corner points. The tilt is the angle between the line joining
        those two points and the north-south line running through whichever of
        the two is more southern.

        Sign convention:
                Positive -> the more southern of the two points is also the more
                        eastern one (image tilts clockwise).
                Negative -> the more southern of the two points is the more
                        western one (image tilts counterclockwise).
        """
        corners = _get_image_corners(image)

        if pass_type == "DESCENDING":
                upper = ee.List(corners.get('north'))
                lower = ee.List(corners.get('west'))
        else:
                upper = ee.List(corners.get('west'))
                lower = ee.List(corners.get('south'))

                

        delta_lon = ee.Number(upper.get(0)).subtract(ee.Number(lower.get(0)))
        delta_lat = ee.Number(upper.get(1)).subtract(ee.Number(lower.get(1)))

        tilt_rad = delta_lat.atan2(delta_lon)  # ee.Number(a).atan2(b) == atan2(a, b)
        tilt_deg = tilt_rad.multiply(180 / math.pi)

        return tilt_deg

def _compute_azimuth(image: ee.Image, pass_type : Literal["ASCENDING", "DESCENDING"]) -> ee.Number:
      """
      Computes the azimuth angle [°] relative to north for an image.
      NOTE: the underlying function rely on assumptions that are valid for
      images of the COPERNICUS/S1_GRD and COPERNICUS/S1GRD_FLOAT collections.
      Transferability to other collections is not guarantied.
      """
      image_grid_azimuth = _get_azimuth_relative_to_image_grid(image.select("angle"))
      image_tilt =  _get_image_tilt(image, pass_type)
      return image_grid_azimuth.add(image_tilt)

def _compute_azimuth_v2(image: ee.Image, pass_type: Literal["ASCENDING", "DESCENDING"]) -> ee.Image:
      """
      Computes the azimuth angle [°] relative to north for an image.
      NOTE: the underlying function rely on assumptions that are valid for
      images of the COPERNICUS/S1_GRD and COPERNICUS/S1GRD_FLOAT collections.
      Transferability to other collections is not guarantied.
      """
      image_grid_azimuth : ee.Image = _get_azimuth_image_relative_to_image_grid(image.select("angle"))
      image_tilt : ee.Number = _get_image_tilt(image, pass_type)
      return image_grid_azimuth.add(ee.Image.constant(image_tilt))

def _compute_lia_and_azimuth(image: ee.Image, pass_type: Literal["ASCENDING", "DESCENDING"]) -> ee.Image:
        """
        returns an image with two bands:
                'LAI': with the projected local incidence angle per pixel
                'AZI': with the average azimuthal angle (constant over all pixels)
        NOTE: The internal logic should be verified again.
        NOTE: I made significant changes to the calculation procedure. I think now it is correct.
        However, the result IS different from previous versions
        """
        srtm = _get_srtm()
        srtm_slope = ee.Terrain.slope(srtm)
        srtm_aspect = ee.Terrain.aspect(srtm)
        azimuth = _compute_azimuth(image, pass_type)
        projected_slope = srtm_slope \
                                        .multiply(
                                        ee.Image.constant(azimuth).subtract(srtm_aspect).multiply(math.pi/180)\
                                                .cos())
        projected_lia = image.select("angle").subtract(projected_slope).abs()
        angles = projected_lia.addBands(azimuth).clip(projected_lia.geometry()) # add azimuth angle to the lia image... this new band is called constant
        angles = angles.select(['angle','constant']).rename(['LIA','AZI']) # REnaming the bands in order to merge it together with the VH and VV datasets (they need to have the same coloumn names in order to)
        return angles

def _compute_lia_and_azimuth_v2(image: ee.Image, pass_type: Literal["ASCENDING", "DESCENDING"]) -> ee.Image:
        """
        returns an image with two bands:
                'LAI': with the projected local incidence angle per pixel
                'AZI': with the average azimuthal angle (constant over all pixels)
        NOTE: The internal logic should be verified again.
        NOTE: I made significant changes to the calculation procedure. I think now it is correct.
        However, the result IS different from previous versions
        """
        srtm = _get_srtm()
        srtm_slope = ee.Terrain.slope(srtm)
        srtm_aspect = ee.Terrain.aspect(srtm)
        azimuth = _compute_azimuth_v2(image, pass_type)
        azimuth.bandNames().getInfo()
        projected_slope = srtm_slope \
                                        .multiply(azimuth.subtract(srtm_aspect).multiply(math.pi/180)\
                                        .cos())
        projected_slope.bandNames().getInfo()
        projected_lia = image.select("angle").subtract(projected_slope).abs()
        projected_lia.bandNames().getInfo()
        angles = projected_lia.addBands(azimuth) # add azimuth angle to the lia image... this new band is called constant
        print(angles.bandNames().getInfo())
        angles = angles.select(['angle','aspect']).rename(['LIA','AZI']) # REnaming the bands in order to merge it together with the VH and VV datasets (they need to have the same coloumn names in order to)
        return angles

    
def _compute_lia_and_azimuth_old(image: ee.Image, pass_type: Literal["ASCENDING", "DESCENDING"]) -> ee.Image:
        """
        returns an image with two bands:
                'LAI': with the projected local incidence angle per pixel
                'AZI': with the average azimuthal angle (constant over all pixels)
        NOTE: The internal logic should be verified again.
        """
        #TODO what is the meaning of the values?
        azimuth_vals = {
               "ASCENDING"      : 270,
               "DESCENDING"     : 360
        }
        rotation_vals = {
               "ASCENDING"      : 180,
               "DESCENDING"     : 180    
        }

        S1angle = image.select('angle')
        #We can use the gradient of the "angle" band of the S1 image to derive the S1 azimuth angle.
        S1_azimuth = ee.Terrain.aspect(S1angle) \
                                .reduceRegion(ee.Reducer.mean(), S1angle.geometry(), 100) \
                                .get('aspect')
        # Attention, this is not actually azimuth, but the look direction across range, which is NOT
        # yet corrected for the angle with which s1_inc is rotated relative to North!!!
        # Calculate True azimuth direction for the near range image edge
        def getCorners(f) -> ee.Element:
                # Get the coords as a transposed array
                coords = ee.Array(f.geometry().coordinates().get(0)).transpose()
                crdLons = ee.List(coords.toList().get(0))
                crdLats = ee.List(coords.toList().get(1))
                minLon = crdLons.sort().get(0)
                maxLon = crdLons.sort().get(-1)
                minLat = crdLats.sort().get(0)
                maxLat = crdLats.sort().get(-1)
                azimuth = ee.Number(crdLons.get(crdLats.indexOf(minLat))).subtract(minLon).atan2(ee.Number(crdLats.get(crdLons.indexOf(minLon))).subtract(minLat)) \
                        .multiply(180.0/math.pi).add(azimuth_vals[pass_type]) 
                return ee.Feature(ee.Geometry.LineString([
                                        crdLons.get(crdLats.indexOf(minLat)),
                                        minLat, minLon, crdLats.get(crdLons.indexOf(minLon))
                                        ]), 
                                  { 'azimuth': azimuth}).copyProperties(f)
        
        azimuthEdge = getCorners(image)
        
        TrueAzimuth = azimuthEdge.get('azimuth')   # This should be some degree off the North direction, due to Earth rotation
        rotationFromNorth_or_South = ee.Number(TrueAzimuth).subtract(rotation_vals[pass_type]) # Use subtract(180.0) for DESCENDING and subtract(360.0) for ASCENDING image.
        
        # Correct the across-range-look direction
        S1_azimuth = ee.Number(S1_azimuth).add(rotationFromNorth_or_South)  
        # Here we derive the terrain slope and aspect
        
        srtm = _get_srtm()
        srtm_slope = ee.Terrain.slope(srtm).select('slope')
        srtm_aspect = ee.Terrain.aspect(srtm).select('aspect')

        # And finally the local incidence angle
        #TODO why use TrueAzimuth in the calculations here, but add S1_azimuth?
        slope_projected2 = srtm_slope.multiply(ee.Image.constant(TrueAzimuth).subtract(90.0).subtract(srtm_aspect).multiply(math.pi/180).cos())
        lia2 = S1angle.subtract(ee.Image.constant(90).subtract(ee.Image.constant(90).subtract(slope_projected2))).abs()
        angles = lia2.addBands(S1_azimuth).clip(lia2.geometry()) # add azimuth angle to the lia image... this new band is called constant
        angles = angles.select(['angle','constant']).rename(['LIA','AZI']) # REnaming the bands in order to merge it together with the VH and VV datasets (they need to have the same coloumn names in order to)
        return angles

def add_lia_and_azimuth(image: ee.Image, pass_type : Literal["ASCENDING", "DESCENDING"]) -> ee.Image:
      return image.addBands(_compute_lia_and_azimuth_old(image, pass_type).copyProperties(image, image.propertyNames()))

In [2]:
my_map = geemap.Map()
my_map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [36]:
pass_type = "ASCENDING"

In [41]:
image = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
  .filterDate('2014-03-01', '2015-05-01')\
  .filter(ee.Filter.eq('instrumentMode', 'IW'))\
  .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))\
  .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))\
  .filter(ee.Filter.eq("orbitProperties_pass", pass_type))\
  .first()
my_map.addLayer(image.select("VV"))
image

In [42]:
within_image_azi = _get_azimuth_relative_to_image_grid(image.select("angle"))
within_image_azi.getInfo()

276.47324725837467

In [43]:
corners = _get_image_corners(image)
corners.getInfo()

{'east': [119.48830141529929, 28.656679835737346],
 'north': [119.17176650242565, 30.27371729091307],
 'south': [116.9790578656085, 28.129987382471363],
 'west': [116.59310773991228, 29.868241014765047]}

In [44]:
corners = _get_image_corners(image)

if pass_type == "DESCENDING":
        upper = ee.List(corners.get('north'))
        lower = ee.List(corners.get('west'))
else:
        upper = ee.List(corners.get('west'))
        lower = ee.List(corners.get('south'))

        

delta_lon = ee.Number(upper.get(0)).subtract(ee.Number(lower.get(0)))
print(delta_lon.getInfo())
delta_lat = ee.Number(upper.get(1)).subtract(ee.Number(lower.get(1)))
print(delta_lat.getInfo())

tilt_rad = delta_lat.atan2(delta_lon)  # ee.Number(a).atan2(b) == atan2(a, b)
tilt_deg = tilt_rad.multiply(180 / math.pi)
print(tilt_deg.getInfo())


-0.3859501256962119
1.7382536322936843
-12.518490209537562


image_tilt = _get_image_tilt(image, "DESCENDING")
image_tilt.getInfo()

pass_type = "DESCENDING"
azimuth = _compute_azimuth(image, pass_type)
azimuth.getInfo()

In [45]:
lia_and_azi = _compute_lia_and_azimuth(image, pass_type)
my_map.addLayer(lia_and_azi.select("LIA"), {}, 'newTry')
lia_and_azi

In [4]:
my_map.addLayer(image.select('VV'), {min: 0.01, max: 0.1}, 'VV')

In [5]:
lia_and_azi = iu._compute_lia_and_azimuth(image)
lia_and_azi

In [6]:
my_map.addLayer(lia_and_azi.select("LIA"), {min: 25, max: 45}, 'LIA')

In [ ]:
lia_and_azi_old = iu._compute_lia_and_azimuth_old(image, pass_type)
lia_and_azi_old

In [8]:
my_map.addLayer(lia_and_azi_old.select("LIA"), {min: 25, max: 45}, 'LIA_old')

In [9]:
lia_and_azi_v2 = iu._compute_lia_and_azimuth_v2(image)
lia_and_azi_v2

['angle', 'aspect']


In [10]:
my_map.addLayer(lia_and_azi_v2.select("LIA"), {min: 25, max: 45}, 'LIA_v2')

lia_and_azi = iu._compute_lia_and_azi(image)
lia_and_azi